In [ ]:
import os
from PIL import Image
from ultralytics import YOLO

def crop_fish_bboxes(
    image_path: str,
    output_dir: str,
    detector,
    device,
    confidence: float = 0.7,
    target_size: tuple = (256, 256),
    min_fish_size: int = 50,
) -> list:
    if image_path.lower().endswith(('.mp4', '.mov', '.avi')):
        return []

    det_out = detector(image_path, conf=confidence)
    if not det_out or not det_out[0].boxes:
        return []

    try:
        pil_img = Image.open(image_path)
    except Exception as e:
        print(f"Не удалось открыть {image_path}: {e}")
        return []

    cropped_paths = []
    for box, conf in zip(det_out[0].boxes.xyxy.cpu().numpy(), det_out[0].boxes.conf.cpu().numpy()):
        if conf < confidence:
            continue

        x1, y1, x2, y2 = map(int, box)
        w, h = x2 - x1, y2 - y1
        if w <= 0 or h <= 0 or min(w, h) < min_fish_size:
            continue

        crop = pil_img.crop((x1, y1, x2, y2)).resize(target_size)

        # --- ФИКС: Конвертируем в RGB, если изображение в RGBA ---
        if crop.mode == 'RGBA':
            crop = crop.convert('RGB')
        # --------------------------------------------------------

        base_name = os.path.splitext(os.path.basename(image_path))[0]
        crop_path = os.path.join(output_dir, f"{base_name}_{x1}_{y1}_{conf:.2f}.jpg")

        try:
            crop.save(crop_path, quality=95)
            cropped_paths.append(crop_path)
        except Exception as e:
            print(f"Не удалось сохранить {crop_path}: {e}")

    return cropped_paths


In [ ]:
input_dir = r"F:\Datasets\Reddit-fishes\aquarium_new\gallery-dl\reddit\Aquariums"
output_dir = r"F:\Datasets\Reddit-fishes\aquarium_new\spade_data\cropped_fish"
os.makedirs(output_dir, exist_ok=True)

for img_name in os.listdir(input_dir):
    img_path = os.path.join(input_dir, img_name)
    crop_fish_bboxes(
        image_path=img_path,
        output_dir=output_dir,
        device='cuda',
        detector=YOLO('./nout.pt'),
        target_size=(256, 256),
    )


In [ ]:
import os
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import numpy as np
from tqdm import tqdm
from PIL import Image
import matplotlib.pyplot as plt
import shutil

# --- Настройки ---
device = 'cuda'
crop_dir = r"F:\Datasets\Reddit-fishes\aquarium_new\spade_data\cropped_fish"

# --- Получаем все пути ---
crop_paths = [
    os.path.join(crop_dir, f)
    for f in os.listdir(crop_dir)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
]

# --- Модель ResNet ---
model = models.resnet18(pretrained=True)
model.fc = torch.nn.Identity()
model.eval().to(device)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# --- Извлекаем эмбеддинги ---
def extract_embeddings(image_paths):
    embeddings = []
    with torch.no_grad():
        for path in tqdm(image_paths, desc="Извлечение эмбеддингов"):
            try:
                img = Image.open(path).convert("RGB")
                img_tensor = transform(img).unsqueeze(0).to(device)
                embedding = model(img_tensor).cpu().numpy().flatten()
                embeddings.append(embedding)
            except Exception as e:
                print(f"Ошибка при обработке {path}: {e}")
    return np.array(embeddings)

# --- Кластеризация ---
embeddings = extract_embeddings(crop_paths)
kmeans = KMeans(n_clusters=2, random_state=42).fit(embeddings)
labels = kmeans.labels_

# --- Визуализация (PCA) ---
pca = PCA(n_components=2)
reduced = pca.fit_transform(embeddings)

plt.figure(figsize=(10, 6))
plt.scatter(reduced[:, 0], reduced[:, 1], c=labels, cmap="viridis", alpha=0.6)
plt.title("Кластеризация кропов (рыбы vs не-рыбы)")
plt.colorbar()
plt.show()

# --- Сохраняем кластеры в отдельные папки ---
os.makedirs("cluster_0", exist_ok=True)
os.makedirs("cluster_1", exist_ok=True)

for path, label in zip(crop_paths, labels):
    cluster_dir = f"cluster_{label}"
    os.makedirs(cluster_dir, exist_ok=True)
    dest_path = os.path.join(cluster_dir, os.path.basename(path))
    shutil.copy(path, dest_path)  # Или используй shutil.copy()


In [ ]:
from IPython.display import display
import random

# Посмотреть случайные изображения из каждого кластера
for cluster_id in [0, 1]:
    print(f"\n--- Кластер {cluster_id} ---")
    cluster_images = [p for p, l in zip(crop_paths, labels) if l == cluster_id]
    for img_path in random.sample(cluster_images, 20):  # 5 случайных изображений
        display(Image.open(img_path))


In [ ]:
import os
from PIL import Image

# Директория с кропами
crop_dir = r"F:\Datasets\Reddit-fishes\aquarium_new\spade_data\cropped_fish"

# Получаем все пути к файлам (например, .jpg, .png)
crop_paths = [
    os.path.join(crop_dir, f)
    for f in os.listdir(crop_dir)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
]

print(f"Найдено {len(crop_paths)} изображений.")


In [ ]:
(r"F:\Datasets\Reddit-fishes\aquarium_new\spade_data\cropped_fish")